[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

© Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es

&copy; José Luis Chiquete Valdivieso. 2017-2026.

# Ejecutar y comprobar un modelo local.

**py521 · IA Generativa en Entornos Locales** · Pythonista® · Instructor: José Luis Chiquete · Demo del Tema 4: *Ecosistema de ejecución local*

En este *notebook* se ejecutará un modelo en el equipo y se leerán las cuatro señales que confirman que funciona en local: 1) memoria usada, 2) tiempo al primer token, 3) tokens por segundo y 4) ejecución local (sin etiqueta de nube y con respuesta sin internet).

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* Funciona con *Ollama* (por omisión) y con cualquier motor con API compatible con *OpenAI*, como *LM Studio* (`LLM_BASE_URL=http://localhost:1234/v1`).
* La memoria y la etiqueta de nube se leen con la API propia de *Ollama*; con otro motor se indica dónde verlas.
* La prueba sin internet (sección 5.2) requiere desconectar la red del equipo a mano.

### Configuración común.

Esta celda es **idéntica en todos los *notebooks***: lee la configuración de variables de entorno (o del archivo `.env`, si existe) y usa valores por omisión cuando no están definidas. Muestra todas las variables, pero **en esta demo importan: `LLM_BASE_URL`, `LLM_MODEL` y, con *Ollama*, `OLLAMA_URL`**. Las claves solo se muestran como «definida / no definida». La tabla completa de variables está en el `README.md` y en `00_preliminares_entorno_y_secretos.ipynb`.

In [ ]:
# Configuración común de py521 (celda idéntica en todos los notebooks).
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)  # las variables del sistema tienen prioridad sobre .env

def _entorno(nombre, por_omision=None):
    valor = os.environ.get(nombre, "").strip()
    return valor or por_omision

# Servicio compatible con OpenAI (Ollama, LM Studio, llama.cpp server, vLLM...).
LLM_BASE_URL = _entorno("LLM_BASE_URL", _entorno("LLM_DEFAULT_BASE_URL", "http://localhost:11434/v1")).rstrip("/")  # el contenedor fija LLM_DEFAULT_BASE_URL
LLM_API_KEY = _entorno("LLM_API_KEY")              # secreto: sin valor por omisión
LLM_MODEL = _entorno("LLM_MODEL", "qwen3:1.7b")
LLM_EMBED_MODEL = _entorno("LLM_EMBED_MODEL", "qwen3-embedding:0.6b")
LLM_MODEL_A = _entorno("LLM_MODEL_A", "qwen3:1.7b-q4_K_M")
LLM_MODEL_B = _entorno("LLM_MODEL_B", "qwen3:1.7b-q8_0")
LLM_CONTEXT_A = int(_entorno("LLM_CONTEXT_A", "4096"))
LLM_CONTEXT_B = int(_entorno("LLM_CONTEXT_B", "16384"))
RAG_DOCS_DIR = Path(_entorno("RAG_DOCS_DIR", "data"))
LLM_PROXY_KEY = _entorno("LLM_PROXY_KEY")          # secreto de la demo 05: sin valor por omisión
LLM_PROXY_HOST = _entorno("LLM_PROXY_HOST", "127.0.0.1")
LLM_PROXY_PORT = int(_entorno("LLM_PROXY_PORT", "8800"))

# API nativa de Ollama (solo para las funciones marcadas «específico de Ollama»).
# Se usa OLLAMA_HOST, como el cliente oficial de Ollama; si no existe, se deriva de LLM_BASE_URL.
def _url_ollama(valor):
    valor = (valor or LLM_BASE_URL.removesuffix("/v1")).strip().rstrip("/")
    if "://" not in valor:
        anfitrion = valor.split("/")[0]
        valor = "http://" + (valor if ":" in anfitrion.split("]")[-1] else valor.replace(anfitrion, anfitrion + ":11434", 1))
    return valor.replace("://0.0.0.0", "://localhost")  # 0.0.0.0 es una dirección de escucha, no de destino

OLLAMA_URL = _url_ollama(_entorno("OLLAMA_HOST"))

for nombre in ["LLM_BASE_URL", "LLM_MODEL", "LLM_EMBED_MODEL", "LLM_MODEL_A", "LLM_MODEL_B",
               "LLM_CONTEXT_A", "LLM_CONTEXT_B", "RAG_DOCS_DIR", "LLM_PROXY_HOST", "LLM_PROXY_PORT", "OLLAMA_URL"]:
    print(f"{nombre:<16} {globals()[nombre]}")
for nombre in ["LLM_API_KEY", "LLM_PROXY_KEY", "HF_TOKEN"]:
    print(f"{nombre:<16} {'definida' if _entorno(nombre) else 'no definida'}")

### Preparación del entorno.

* Las siguientes celdas crearán el cliente de la API compatible con *OpenAI*, comprobarán que los modelos están disponibles y definirán las funciones de la demo; no hace falta leerlas.

In [ ]:
from openai import OpenAI

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY or "sin-clave")  # Ollama y LM Studio sin autenticación ignoran la clave

def comprobar_servicio(modelos):
    """Comprueba que el servicio responde (GET /models) y que ofrece los modelos indicados."""
    try:
        disponibles = {m.id for m in cliente.models.list().data}
    except Exception as error:
        raise RuntimeError(f"No responde el servicio en {LLM_BASE_URL} ({type(error).__name__}). "
                           "Abre Ollama o el motor que uses, o revisa LLM_BASE_URL.") from None
    faltan = [m for m in modelos if m not in disponibles and f"{m}:latest" not in disponibles]
    if faltan:
        raise RuntimeError(f"El servicio no ofrece: {', '.join(faltan)}. En Ollama: "
                           + " · ".join(f"ollama pull {m}" for m in faltan))
    print(f"Servicio en {LLM_BASE_URL}: responde. Modelos: {', '.join(modelos)}.")

comprobar_servicio([LLM_MODEL])

In [ ]:
import time

def preguntar(prompt, modelo=None, temperatura=None, razonamiento=None, max_tokens=None, sistema=None):
    """Envía una petición a /chat/completions en modo streaming y mide los tiempos.

    razonamiento: None (valor del modelo), "none" (desactivado) o "low" / "medium" / "high".
    Se envía como `reasoning_effort` (Ollama, vLLM, LM Studio) y como
    `chat_template_kwargs.enable_thinking` (llama.cpp server, vLLM con Qwen3); cada motor ignora lo que no usa.
    """
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": prompt}]
    extra, argumentos = {}, {}
    if temperatura is not None:
        argumentos["temperature"] = temperatura
    if max_tokens is not None:
        argumentos["max_tokens"] = max_tokens
    if razonamiento is not None:
        argumentos["reasoning_effort"] = razonamiento
        extra["chat_template_kwargs"] = {"enable_thinking": razonamiento != "none"}
    inicio = time.perf_counter()
    primer_token = primer_texto = None
    texto, pensado, uso, fin = [], [], None, None
    flujo = cliente.chat.completions.create(model=modelo or LLM_MODEL, messages=mensajes, stream=True,
                                            stream_options={"include_usage": True}, extra_body=extra or None,
                                            **argumentos)
    for parte in flujo:
        if parte.usage:
            uso = parte.usage
        if not parte.choices:
            continue
        delta = parte.choices[0].delta
        extras = delta.model_extra or {}
        trozo_razonamiento = extras.get("reasoning") or extras.get("reasoning_content")  # el nombre varía por motor
        ahora = time.perf_counter() - inicio
        if primer_token is None and (delta.content or trozo_razonamiento):
            primer_token = ahora
        if primer_texto is None and delta.content:
            primer_texto = ahora
        pensado.append(trozo_razonamiento or "")
        texto.append(delta.content or "")
        fin = parte.choices[0].finish_reason or fin
    total = time.perf_counter() - inicio
    tokens = uso.completion_tokens if uso else 0
    generacion = total - (primer_token or total)
    return {
        "respuesta": "".join(texto).strip(),
        "razonamiento": "".join(pensado).strip(),
        "primer_token_s": primer_token,
        "primer_texto_s": primer_texto,
        "total_s": total,
        "tokens": tokens,
        "tokens_s": tokens / generacion if generacion > 0 else 0.0,
        "fin": fin,
    }

In [ ]:
# ── 🦙 Específico de Ollama ────────────────────────────────────────────────
# Estas funciones usan la API nativa de Ollama (OLLAMA_URL). Con otro motor
# devuelven None y muestran un aviso; el resto del notebook sigue funcionando.
import httpx

def ollama_version():
    """Versión de Ollama, o None si OLLAMA_URL no es un servicio Ollama."""
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=5).json()["version"]
    except Exception:
        return None

ES_OLLAMA = ollama_version() is not None
print(f"Ollama en {OLLAMA_URL}:", f"sí (versión {ollama_version()})" if ES_OLLAMA
      else "no detectado; las funciones específicas de Ollama se omitirán")

_avisos_mostrados = set()

def aviso_no_ollama(que, equivalente):
    """Muestra una sola vez por notebook que una medición es específica de Ollama y su equivalente."""
    if que not in _avisos_mostrados:
        _avisos_mostrados.add(que)
        print(f"ℹ️ {que} solo se mide con Ollama. Equivalente en otros motores: {equivalente}")

def ollama_memoria(modelo):
    """🦙 Memoria del modelo cargado según /api/ps (total, parte en VRAM y ventana de contexto)."""
    if not ES_OLLAMA:
        aviso_no_ollama("La memoria ocupada", "LM Studio: indicador de memoria de la aplicación; llama.cpp server: "
                        "registro al cargar el modelo; vLLM: `nvidia-smi` o el registro de arranque; "
                        "en todos: el monitor del sistema.")
        return None
    for m in httpx.get(f"{OLLAMA_URL}/api/ps", timeout=10).json().get("models", []):
        if m["name"] == modelo or m["model"] == modelo:
            return {"total_gb": m["size"] / 1e9, "vram_gb": m.get("size_vram", 0) / 1e9,
                    "contexto": m.get("context_length")}
    return None

def ollama_descargar(modelo):
    """🦙 Saca el modelo de la memoria (keep_alive = 0) para medir la siguiente carga desde cero."""
    if ES_OLLAMA:
        httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": modelo, "keep_alive": 0}, timeout=60)


def ollama_es_nube(modelo):
    """🦙 True si el modelo se ejecuta en la nube de Ollama (etiqueta «cloud» o servidor remoto)."""
    if not ES_OLLAMA:
        aviso_no_ollama("La etiqueta de nube", "LM Studio, llama.cpp server y vLLM ejecutan en el equipo que "
                        "los aloja; comprueba que LLM_BASE_URL apunta a ese equipo.")
        return None
    info = httpx.post(f"{OLLAMA_URL}/api/show", json={"model": modelo}, timeout=30).json()
    return modelo.endswith("cloud") or "-cloud" in modelo or bool(info.get("remote_host"))

In [ ]:
import socket

import matplotlib.pyplot as plt

def hay_internet(destino=("ollama.com", 443), espera=3):
    """True si el equipo alcanza un servidor de internet."""
    try:
        socket.create_connection(destino, timeout=espera).close()
        return True
    except OSError:
        return False

## 1. Cargar el modelo y hacer tres peticiones.

* La siguiente celda sacará primero el modelo de la memoria (🦙 solo con *Ollama*), para que la primera petición incluya la carga del modelo.
* Después hará tres peticiones; cada una cambia la primera línea para que cada medición empiece desde cero (el motor guarda lo que ya procesó y respondería al instante).
* Las secciones 2 a 4 leen las señales sobre estas tres peticiones.

In [ ]:
PROMPT = "Explica en tres frases qué es una red de computadoras."

ollama_descargar(LLM_MODEL)
mediciones = []
for i in range(1, 4):
    r = preguntar(f"Petición {i}.\n{PROMPT}", razonamiento="none", temperatura=0.0, max_tokens=200)
    mediciones.append(r)
    print(f"Petición {i}: primer token {r['primer_token_s']:.2f} s · {r['tokens_s']:.1f} tokens/s · {r['tokens']} tokens")
print("\nRespuesta:", mediciones[-1]["respuesta"])

## 2. Señal 1: memoria usada.

* En *Ollama*, `/api/ps` informa la memoria total del modelo cargado, cuánta está en la GPU (VRAM) y la ventana de contexto.
* Sin GPU, el modelo ocupa memoria RAM del sistema. En una Mac, la memoria unificada es compartida.
* Sin GPU, mira solo el total.

In [ ]:
memoria = ollama_memoria(LLM_MODEL)
if memoria:
    en_gpu = memoria["vram_gb"] / memoria["total_gb"] * 100 if memoria["total_gb"] else 0
    print(f"Modelo cargado: {memoria['total_gb']:.2f} GB en total · {memoria['vram_gb']:.2f} GB en VRAM ({en_gpu:.0f} %) · "
          f"ventana de contexto: {memoria['contexto']} tokens")

* Compara la cifra con el monitor del sistema: Administrador de tareas (Windows), Monitor de actividad (macOS) o el monitor del sistema (Linux).

## 3. Señal 2: tiempo al primer token.

* Es la espera antes de que empiece la respuesta.
* La primera petición tarda más porque incluye la carga del modelo en memoria; las siguientes lo encuentran cargado.

In [ ]:
etiquetas = [f"Petición {i}" for i in range(1, len(mediciones) + 1)]
plt.figure(figsize=(5, 3.2))
plt.bar(etiquetas, [m["primer_token_s"] for m in mediciones], color="#4C72B0")
plt.title("Tiempo al primer token (s)")
plt.tight_layout()
plt.show()

## 4. Señal 3: tokens por segundo.

* Es la velocidad con que aparece la respuesta una vez que empieza. Depende del equipo (CPU o GPU) y del tamaño del modelo.

In [ ]:
plt.figure(figsize=(5, 3.2))
plt.bar(etiquetas, [m["tokens_s"] for m in mediciones], color="#55A868")
plt.title("Tokens por segundo")
plt.tight_layout()
plt.show()

## 5. Señal 4: ejecución local.

### 5.1 El modelo no lleva etiqueta de nube.

* En *Ollama*, algunos modelos se ejecutan en su nube: la petición sale del equipo. Esos modelos llevan «cloud» en el nombre.
* La siguiente celda clasificará los modelos que ofrece el servicio y comprobará el modelo elegido.

In [ ]:
modelos = sorted(m.id for m in cliente.models.list().data)
for nombre in modelos:
    nube = ollama_es_nube(nombre) if ES_OLLAMA else None
    print(f"{nombre:<32}", {True: "nube ☁️", False: "local 💻", None: "—"}[nube])

nube_elegido = ollama_es_nube(LLM_MODEL)
print(f"\nModelo elegido: {LLM_MODEL} →",
      {True: "NUBE: la petición sale del equipo", False: "local", None: "no comprobable con este motor"}[nube_elegido])

* Para desactivar las funciones de nube de *Ollama*, se define `OLLAMA_NO_CLOUD=1` en el servicio o `"disable_ollama_cloud": true` en `~/.ollama/server.json`, y se reinicia ([documentación oficial](https://docs.ollama.com/faq)).

### 5.2 Respuesta sin internet.

1. Ejecuta la siguiente celda **con** internet: debe indicar «con internet».
2. Desconecta la red del equipo (wifi y cable).
3. Vuelve a ejecutar la celda: debe indicar «sin internet» y aun así mostrar una respuesta.

Si llega la respuesta sin internet, el modelo corre en el equipo. Con un modelo de nube, la petición falla. Si `LLM_BASE_URL` apunta a otro equipo de la red local, desconecta solo la salida a internet, no la red local.

> En una sesión en línea no desconectes el equipo que transmite: haz la prueba antes y muestra la captura, o usa un segundo equipo.

In [ ]:
conectado = hay_internet()
print("Estado de la red:", "con internet 🌐" if conectado else "sin internet 🔌")
r = preguntar("Comprobación sin red.\nResponde solo con la palabra: listo", razonamiento="none", temperatura=0.0, max_tokens=10)
print(f"Respuesta del modelo: {r['respuesta']} ({r['total_s']:.1f} s)")

## 6. Alternativa: *LM Studio*.

*LM Studio* ([lmstudio.ai](https://lmstudio.ai)) es una aplicación de escritorio con interfaz gráfica. Su servicio local usa la misma API compatible con *OpenAI*.

1. En *LM Studio*, descarga un modelo pequeño (por ejemplo, *Qwen3 1.7B*) y cárgalo.
2. En **Developer**, activa **Start server** (dirección por omisión: `http://localhost:1234/v1`).
3. Define `LLM_BASE_URL=http://localhost:1234/v1` y `LLM_MODEL` con el identificador que muestra *LM Studio*, y reinicia el *kernel*.

Con *LM Studio*, las secciones 2 y 5.1 muestran un aviso; el tiempo al primer token, los tokens por segundo y la prueba sin internet funcionan igual. *LM Studio* también muestra la velocidad y la memoria en su propia interfaz.

---
## ✅ Cierre.

| Señal | Cómo se comprobó |
|---|---|
| 1. Memoria usada | `/api/ps` (*Ollama*) o monitor del sistema |
| 2. Tiempo al primer token | Espera hasta el primer token; la primera petición incluye la carga |
| 3. Tokens por segundo | Velocidad de la respuesta una vez que empieza |
| 4. Ejecución local | Sin etiqueta de nube (*Ollama*) y respuesta con la red desconectada |

**Siguiente *notebook*:** `03_cuantizacion_y_ventana_de_contexto.ipynb`.

<p style="text-align: center"><a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>